# VetXRay Dataset — Minimal loading example

This notebook shows how to load a single chest X-ray from the **VetXRay** dataset,
retrieve its annotation from the metadata spreadsheet, apply basic image
preprocessing, and display the result.

**Install dependencies** (run once):
```bash
pip install pydicom pandas numpy matplotlib openpyxl
```


In [ ]:
import os
import pydicom
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Configuration ──────────────────────────────────────────────────────────────
# Path to the folder containing all .dcm files
DICOM_DIR = None # TODO: set this to the path where your DICOM files are located

# Path to the annotation spreadsheet (same folder as this notebook)
XLSX_PATH = None # TODO: set this to the path of the annotation spreadsheet (e.g., "annotations.xlsx")

# Which file to load (must exist in DICOM_DIR and in the spreadsheet)
SAMPLE_FILE = "IM-0015-0001-0001.dcm" # Example filename, change as needed


In [ ]:
# ── 1. Load the DICOM file ────────────────────────────────────────────────────
ds = pydicom.dcmread(os.path.join(DICOM_DIR, SAMPLE_FILE))

# Raw pixel data as a 2-D NumPy array (uint16)
pixels = ds.pixel_array

print(f"Loaded:   {SAMPLE_FILE}")
print(f"Shape:    {pixels.shape}  |  dtype: {pixels.dtype}")
print(f"Modality: {getattr(ds, 'Modality', 'N/A')}")

# ── 2. Look up the annotation row ─────────────────────────────────────────────
annotations = pd.read_excel(XLSX_PATH, engine="openpyxl")
row = annotations.loc[annotations["FileName"] == SAMPLE_FILE].iloc[0]

species    = row["specie"]
breed      = row["breed"]
projection = row["Projection"]
quality    = row["Quality"]
# TAG is pipe-separated; "no_finding" means a healthy image
findings   = [t.strip() for t in str(row["TAG"]).split("|") if t.strip()]

print(f"\nAnnotation:")
print(f"  Species / Breed : {species} / {breed}")
print(f"  Projection      : {projection}")
print(f"  Quality         : {quality}")
print(f"  Findings        : {', '.join(findings)}")

# ── 3. Preprocess: percentile contrast stretch ────────────────────────────────
img = pixels.astype(np.float32)
p_low, p_high = np.percentile(img, [2, 98])
img = np.clip(img, p_low, p_high)
img = (img - p_low) / (p_high - p_low)   # now in [0, 1]

# MONOCHROME1: high pixel value = dark on film → invert so lungs appear dark
if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
    img = 1.0 - img


In [ ]:
# ── 4. Display ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 7))

ax.imshow(img, cmap="gray", aspect="equal")
ax.axis("off")

ax.set_title(
    f"{species}  ·  {breed}  ·  Projection: {projection}",
    fontsize=12, fontweight="bold", pad=10,
)

finding_text = ", ".join(findings) if findings else "—"
info = f"Quality: {quality}\nFindings: {finding_text}"
ax.text(
    0.5, -0.02, info,
    transform=ax.transAxes,
    ha="center", va="top",
    fontsize=9, color="#333333",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#f5f5f5", edgecolor="#cccccc"),
)

plt.tight_layout()
plt.show()
